<table>
  <tr>
    <td style="text-align: center">
      <a href="https://colab.research.google.com/github/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/05_BigQuery_Property_Graph.ipynb">
        <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Farvind-dhariwal%2Faeon-credit-gcp-workshop%2Fmain%2Ftrack1_platform_governance%2Fnotebook%2F05_BigQuery_Property_Graph.ipynb">
        <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/notebook/05_BigQuery_Property_Graph.ipynb">
        <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/05_BigQuery_Property_Graph.ipynb">
        <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/05_BigQuery_Property_Graph.ipynb">
        <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
      </a>
    </td>
  </tr>
</table>

# Track 1 (Notebook 05 — Lab 1.5): BigQuery Property Graph (ISO GQL & Interactive Graph Visualization)
**AEON Credit Service Malaysia (ACSM) — Google Agentic Data Cloud Workshop (`asia-southeast1` Singapore)**

<p align="center">
  <img src="https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/images/notebook5_bq_graph_and_bqca_flow.png" alt="Track 1 Notebook 05 Architecture: BigQuery Property Graph (ISO GQL)" width="95%">
</p>

---

### 📋 Notebook 05 Step-by-Step Execution Summary
1. **Step 0 (Environment Setup, `bigquery-magics>=0.12.1` & APIs)**: Auto-detect active `PROJECT_ID` and Singapore region (`asia-southeast1`), install `bigquery-magics>=0.12.1` to unlock interactive `%%bigquery --graph` visualizations, and enable `bigquery.googleapis.com`, `bigqueryreservation.googleapis.com`, and `dataplex.googleapis.com`.
2. **Step 1 (Provision BigQuery Enterprise Reservation & Invoke `.sql` Bootstrap Files)**:
   - **Step 1.0**: Provision a BigQuery **Enterprise** reservation (`acsm-graph-enterprise-res`, `baseline=0`, `max_slots=500`) and assign it to `projects/{PROJECT_ID}` (`jobType=QUERY`), required for `CREATE OR REPLACE PROPERTY GRAPH` and `GRAPH_TABLE` ISO GQL queries.
   - **Step 1.1**: Ensure `acsm_bronze`, `acsm_silver`, and `acsm_gold` exist (so this notebook can run standalone) and execute [`06_bigquery_graph_and_bqca.sql`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/sql/06_bigquery_graph_and_bqca.sql) to materialize the 4 Graph Node Tables + 3 Graph Edge Tables and `CREATE OR REPLACE PROPERTY GRAPH acsm_gold.acsm_credit_ecosystem_graph`.
3. **Step 2 (Inspect BigQuery Property Graph Schema & Cardinality)**: Audit the nodes (`Customer`, `CreditFacility`, `Merchant`, `EmployerSegment`) and edges (`HOLDS_FACILITY`, `TRANSACTED_AT`, `WORKS_IN_SEGMENT`) of `acsm_gold.acsm_credit_ecosystem_graph`.
4. **Step 3 (ISO GQL Query 1 — 2-Hop Customer $\rightarrow$ CreditFacility & Merchant Portfolio Traversal)**: Run `%%bigquery --graph` (`RETURN TO_JSON(p) AS path`) for interactive node-and-edge visualization and `GRAPH_TABLE(...)` to profile multi-product customers across Malaysian states.
5. **Step 4 (ISO GQL Query 2 — Multi-Hop First-Party Fraud & Delinquency Contagion Ring Detection)**: Traverse cyclical diamond rings `(c1:Customer)-[:TRANSACTED_AT]->(m:Merchant)<-[:TRANSACTED_AT]-(c2:Customer)` + `(c1)-[:WORKS_IN_SEGMENT]->(e:EmployerSegment)<-[:WORKS_IN_SEGMENT]-(c2)` where both customers have active delinquency (`is_delinquent = TRUE`) using `%%bigquery --graph` and `GRAPH_TABLE(...)`.
6. **Step 5 (ISO GQL Query 3 — AEON Ecosystem Cross-Sell Path Discovery)**: Traverse from pre-qualified Easy Payment (`EP`) customers with zero delinquency, low DSR, and active PDPA consent (`pdpa_marketing_consent = 'Y'`) who hold zero active Credit Cards (`active_card_count = 0`) through `TRANSACTED_AT` edges to identify top AEON Privilege Merchants for instant Credit Card cross-sell.
7. **Step 6 (Reservation Cleanup)**: Unassign the project and delete the `acsm-graph-enterprise-res` reservation so your project reverts to standard On-Demand query pricing before proceeding to **Notebook 06 (`06_BigQuery_Conversational_Analytics_Agent.ipynb`)**.

---
## Step 0: Configure Parameters (`PROJECT_ID` & Singapore Region), Install `bigquery-magics>=0.12.1` for `--graph` Visualization & Enable APIs
This notebook dynamically detects your active Google Cloud project (`PROJECT_ID`), runs in **`asia-southeast1` (Singapore)**, and upgrades **`bigquery-magics>=0.12.1`** to enable native interactive node-and-edge graph visualizations (`%%bigquery --graph` returning `TO_JSON(p) AS path`).

In [ ]:
# @title Step 0: Auto-Detect `PROJECT_ID`, Install `bigquery-magics>=0.12.1` (`--graph`), & Enable APIs
import os
import subprocess
import google.auth

PROJECT_ID = ""  # @param {type:"string"}
LOCATION = "asia-southeast1"  # @param {type:"string"}
REPO_URL = "https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop.git"
REPO_DIR = "aeon-credit-gcp-workshop"

if not PROJECT_ID or PROJECT_ID == "<PROJECT_ID>":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip()
if not PROJECT_ID:
    PROJECT_ID = subprocess.check_output(
        ["gcloud", "config", "get-value", "project"], text=True
    ).strip()
if not PROJECT_ID or PROJECT_ID == "(unset)":
    _, PROJECT_ID = google.auth.default()

BUCKET_NAME = f"acsm-workshop-landing-{PROJECT_ID}"
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["LOCATION"] = LOCATION
os.environ["BUCKET_NAME"] = BUCKET_NAME

# Install/upgrade bigquery-magics >= 0.12.1 required for interactive `%%bigquery --graph` visualization
!pip install -q "bigquery-magics>=0.12.1"
%reload_ext bigquery_magics
import bigquery_magics
bigquery_magics.context.project = PROJECT_ID
bigquery_magics.context.location = LOCATION

!gcloud config set project $PROJECT_ID
!gcloud services enable \
    bigquery.googleapis.com \
    bigqueryreservation.googleapis.com \
    dataplex.googleapis.com \
    storage.googleapis.com \
    --project=$PROJECT_ID

![ -d aeon-credit-gcp-workshop ] || git clone --depth 1 $REPO_URL $REPO_DIR
!git -C $REPO_DIR pull --ff-only
!gcloud storage buckets describe gs://{BUCKET_NAME} >/dev/null 2>&1 || gcloud storage buckets create gs://{BUCKET_NAME} --location={LOCATION} --uniform-bucket-level-access
!gcloud storage cp aeon-credit-gcp-workshop/data/full_compressed/*.csv.gz gs://{BUCKET_NAME}/full_compressed/

print(f"✅ Active Project        : {PROJECT_ID}")
print(f"✅ BigQuery Region       : {LOCATION} (Singapore)")
print(f"✅ BigQuery Graph Magics : bigquery-magics>=0.12.1 loaded (`%%bigquery --graph` enabled)")

---
## Step 1: Provision BigQuery Enterprise Reservation (`0` Baseline / `500` Max Slots) & Invoke `.sql` Bootstrap Files
Running **BigQuery Property Graph (`CREATE OR REPLACE PROPERTY GRAPH` and `GRAPH_TABLE` ISO GQL queries)** requires a **BigQuery Enterprise Edition** reservation assigned to your project for `QUERY` execution:
1. **Step 1.0**: Creates an **Enterprise** reservation (`acsm-graph-enterprise-res`) in `asia-southeast1` with **Baseline Slots = `0`** (so zero slots are billed when idle) and **Max Autoscale Slots = `500`**, and assigns it to `projects/{PROJECT_ID}` (`jobType=QUERY`). *(In **Step 6** at the end of this notebook, we delete both the assignment and the reservation as part of cleanup).*
2. **Step 1.1**: Executes [`track1_platform_governance/sql/06_bigquery_graph_and_bqca.sql`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/sql/06_bigquery_graph_and_bqca.sql) to build the Property Graph Node & Edge tables and `CREATE OR REPLACE PROPERTY GRAPH acsm_gold.acsm_credit_ecosystem_graph`.

> **Standalone Execution Support**: If `acsm_bronze` or `acsm_silver` have not been populated yet in this project, Step 1.1 automatically executes `00_create_8_tables_ddl_with_descriptions.sql`, `01_load_bronze_layer_from_gcs.sql`, and `02_medallion_silver_and_gold_layer.sql` first.

In [ ]:
# @title Step 1.0: Create BigQuery Enterprise Reservation (Baseline=0, Max=500 Slots) & Assign to Project (`bq_reservation_helper.py`)
%run aeon-credit-gcp-workshop/track1_platform_governance/scripts/bq_reservation_helper.py setup --project $PROJECT_ID --location $LOCATION --reservation-id acsm-graph-enterprise-res --baseline-slots 0 --max-slots 500

In [ ]:
# @title Step 1.1: Invoke `.sql` Files to Build `acsm_gold` Semantic Marts & `acsm_credit_ecosystem_graph`
!bq show --project_id=$PROJECT_ID $PROJECT_ID:acsm_bronze.Fact_CC_Sales >/dev/null 2>&1 || ( \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/00_create_8_tables_ddl_with_descriptions.sql && \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/01_load_bronze_layer_from_gcs.sql \
)
!bq show --project_id=$PROJECT_ID $PROJECT_ID:acsm_gold.gold_aeon_customer360_profile >/dev/null 2>&1 || ( \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/02_medallion_silver_and_gold_layer.sql \
)
!bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/06_bigquery_graph_and_bqca.sql
print("✅ Executed 06_bigquery_graph_and_bqca.sql (`acsm_gold.acsm_credit_ecosystem_graph` & Gold BQCA Marts ready)!")

---
## Step 2 (Part A.1): Inspect the BigQuery Property Graph Schema & Node/Edge Cardinality
**BigQuery Property Graph (`CREATE OR REPLACE PROPERTY GRAPH`)** allows ACSM to model complex many-to-many financial relationships directly on top of governed BigQuery tables in `acsm_gold`—with **zero ETL export to external graph databases** and full inheritance of BigQuery's IAM, Dataplex lineage, and BNM RMiT regional governance (`asia-southeast1`).

### Graph Topology: `acsm_gold.acsm_credit_ecosystem_graph`
| Graph Element Type | Label | Underlying `acsm_gold` Table | Key Identifier(s) | Business Role in ACSM Ecosystem |
|---|---|---|---|---|
| **Node** | `Customer` | `graph_node_customer` | `KEY (cif_id)` | Deduplicated `m3CIF` customer profile with state, income, `wallet_tier`, `akpk_status`, `combined_unpaid_osp_myr`, and `is_delinquent` flag |
| **Node** | `CreditFacility` | `graph_node_credit_facility` | `KEY (facility_id)` | Easy Payment (`EP_*`) and Credit Card (`CC_*`) facilities with `credit_score`, `score_rank`, `new_dsr`, and `exposure_amount_myr` |
| **Node** | `Merchant` | `graph_node_merchant` | `KEY (merchant_id)` | Retail spend locations (`LDESC`) and `PriviledgeMerchantsGrp` across `Fact_CC_Sales` and `Fact_EP_Sales` |
| **Node** | `EmployerSegment` | `graph_node_employer_segment` | `KEY (employer_segment_id)` | Employer / Nature-of-Business (`NOB`) + State clusters (`NOB\|State`) |
| **Edge** | `HOLDS_FACILITY` | `graph_edge_holds_facility` | `(Customer)-[HOLDS_FACILITY]->(CreditFacility)` | Links customers to their approved/applied EP and CC credit facilities |
| **Edge** | `TRANSACTED_AT` | `graph_edge_transacted_at` | `(Customer)-[TRANSACTED_AT]->(Merchant)` | Links customers to retail merchants with aggregated `total_spend_myr` and `tx_count` |
| **Edge** | `WORKS_IN_SEGMENT` | `graph_edge_works_in_segment` | `(Customer)-[WORKS_IN_SEGMENT]->(EmployerSegment)` | Links customers to shared employment/industry and state segments |

> 🕸️ **Interactive Node-and-Edge Graph Visualization in This Notebook (`%%bigquery --graph`)**:
> Using **`bigquery-magics>=0.12.1`**, any cell starting with `%%bigquery --graph` that queries `GRAPH acsm_gold.acsm_credit_ecosystem_graph`, matches a path `p = (node1)-[rel]->(node2)`, and returns **`TO_JSON(p) AS path`** renders an **interactive Graph Visualization widget** directly inside the notebook output ([Official BigQuery Graph Visualization Docs](https://docs.cloud.google.com/bigquery/docs/graph-visualization)):
> - **Click any Node or Edge** to open the side panel displaying its labels and properties (`cif_id`, `net_income_myr`, `product_line`, `merchant_group`, `total_spend_myr`, etc.).
> - **Switch Layouts** in the top-left menu between **Force layout** (default), **Hierarchical**, and **Sequential**.
> - **Toggle Schema View** (`Show schema`) to inspect the underlying node/edge schema topology.
> - **Right-click any Node** to **Expand**, **Collapse**, or **Show neighbors**.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Audit Node and Edge Cardinality of `acsm_gold.acsm_credit_ecosystem_graph`
SELECT '1. NODE: Customer' AS graph_element, 'acsm_gold.graph_node_customer' AS backing_table, COUNT(*) AS row_count FROM `acsm_gold.graph_node_customer`
UNION ALL
SELECT '2. NODE: CreditFacility', 'acsm_gold.graph_node_credit_facility', COUNT(*) FROM `acsm_gold.graph_node_credit_facility`
UNION ALL
SELECT '3. NODE: Merchant', 'acsm_gold.graph_node_merchant', COUNT(*) FROM `acsm_gold.graph_node_merchant`
UNION ALL
SELECT '4. NODE: EmployerSegment', 'acsm_gold.graph_node_employer_segment', COUNT(*) FROM `acsm_gold.graph_node_employer_segment`
UNION ALL
SELECT '5. EDGE: HOLDS_FACILITY (Customer -> CreditFacility)', 'acsm_gold.graph_edge_holds_facility', COUNT(*) FROM `acsm_gold.graph_edge_holds_facility`
UNION ALL
SELECT '6. EDGE: TRANSACTED_AT (Customer -> Merchant)', 'acsm_gold.graph_edge_transacted_at', COUNT(*) FROM `acsm_gold.graph_edge_transacted_at`
UNION ALL
SELECT '7. EDGE: WORKS_IN_SEGMENT (Customer -> EmployerSegment)', 'acsm_gold.graph_edge_works_in_segment', COUNT(*) FROM `acsm_gold.graph_edge_works_in_segment`
ORDER BY graph_element;

---
## Step 3 (Part A.2 — ISO GQL Query 1): 2-Hop Customer $\rightarrow$ CreditFacility & Merchant Portfolio Traversal (`%%bigquery --graph`)
Using standard **ISO GQL (`GRAPH`, `MATCH p = ...`, `RETURN TO_JSON(p) AS path`)**, we traverse 2 hops simultaneously from a `Customer` node to both their `CreditFacility` (`EP` or `CC`) and the `Merchant` locations where they spend:
```
p = (f:CreditFacility) <-[h:HOLDS_FACILITY]- (c:Customer) -[t:TRANSACTED_AT]-> (m:Merchant)
```
Below:
1. **Step 3.1 (`%%bigquery --graph`)** renders the **interactive Node-and-Edge Graph Visualization** directly in the notebook output so you can visually explore `Customer`, `CreditFacility`, and `Merchant` nodes and their connecting edges (`HOLDS_FACILITY`, `TRANSACTED_AT`).
2. **Step 3.2 (`GRAPH_TABLE`)** aggregates those 2-hop paths into a tabular executive summary by **Malaysian State**, **Product Line**, and **Privilege Merchant Group**.

In [ ]:
%%bigquery --graph
-- Step 3.1: Interactive Graph Visualization of 2-Hop (CreditFacility)<-[HOLDS_FACILITY]-(Customer)-[TRANSACTED_AT]->(Merchant) Paths
GRAPH acsm_gold.acsm_credit_ecosystem_graph
MATCH p = (f:CreditFacility)<-[h:HOLDS_FACILITY]-(c:Customer)-[t:TRANSACTED_AT]->(m:Merchant)
WHERE c.is_delinquent = FALSE
  AND t.total_spend_myr >= 1500
RETURN TO_JSON(p) AS path
LIMIT 40;

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Step 3.2: Tabular Aggregation of 2-Hop Traversal across (CreditFacility)<-[HOLDS_FACILITY]-(Customer)-[TRANSACTED_AT]->(Merchant)
SELECT
  state,
  product_line,
  merchant_group,
  COUNT(DISTINCT cif_id) AS active_customers,
  COUNT(DISTINCT facility_id) AS credit_facilities_held,
  ROUND(AVG(new_dsr), 2) AS avg_facility_dsr_pct,
  ROUND(SUM(exposure_amount_myr), 2) AS total_credit_exposure_myr,
  ROUND(SUM(total_spend_myr), 2) AS total_merchant_spend_myr,
  COUNTIF(is_delinquent) AS delinquent_customers_in_path
FROM GRAPH_TABLE(
  `acsm_gold.acsm_credit_ecosystem_graph`
  MATCH (f:CreditFacility)<-[h:HOLDS_FACILITY]-(c:Customer)-[t:TRANSACTED_AT]->(m:Merchant)
  COLUMNS (
    c.cif_id AS cif_id,
    c.state AS state,
    c.is_delinquent AS is_delinquent,
    f.facility_id AS facility_id,
    f.product_line AS product_line,
    f.exposure_amount_myr AS exposure_amount_myr,
    f.new_dsr AS new_dsr,
    t.total_spend_myr AS total_spend_myr,
    m.merchant_group AS merchant_group
  )
)
GROUP BY state, product_line, merchant_group
ORDER BY total_merchant_spend_myr DESC
LIMIT 15;

---
## Step 4 (Part A.3 — ISO GQL Query 2): Multi-Hop First-Party Fraud & Delinquency Contagion Ring Detection (`%%bigquery --graph`)
In consumer credit and installment financing, **first-party synthetic fraud and merchant-collusion rings** often manifest as multiple delinquent borrowers (`is_delinquent = TRUE`) from the same **Employer/Industry Segment (`EmployerSegment`)** funneling high-value purchases through the same **Merchant (`Merchant`)**:
```
p1 = (c1:Customer)-[t1:TRANSACTED_AT]->(m:Merchant)<-[t2:TRANSACTED_AT]-(c2:Customer)
p2 = (c1)-[w1:WORKS_IN_SEGMENT]->(e:EmployerSegment)<-[w2:WORKS_IN_SEGMENT]-(c2)
```
Writing this as a multi-way self-join in relational SQL is error-prone, whereas **ISO GQL `MATCH`** expresses the cyclical diamond pattern declaratively:
1. **Step 4.1 (`%%bigquery --graph`)** returns `TO_JSON(p1) AS merchant_ring_path, TO_JSON(p2) AS employer_ring_path` so you can visually inspect the **cyclical diamond contagion rings** (delinquent `Customer` pairs linked via both a shared `Merchant` node and a shared `EmployerSegment` node) directly in the interactive notebook graph viewer.
2. **Step 4.2 (`GRAPH_TABLE`)** ranks the top connected delinquent rings by pair count and combined unpaid balance (`pair_unpaid_osp_myr`).

In [ ]:
%%bigquery --graph
-- Step 4.1: Interactive Graph Visualization of Delinquency Contagion & Collusion Rings (Shared Merchant + Shared EmployerSegment)
GRAPH acsm_gold.acsm_credit_ecosystem_graph
MATCH p1 = (c1:Customer)-[t1:TRANSACTED_AT]->(m:Merchant)<-[t2:TRANSACTED_AT]-(c2:Customer),
      p2 = (c1)-[w1:WORKS_IN_SEGMENT]->(e:EmployerSegment)<-[w2:WORKS_IN_SEGMENT]-(c2)
WHERE c1.cif_id < c2.cif_id
  AND c1.is_delinquent = TRUE
  AND c2.is_delinquent = TRUE
RETURN TO_JSON(p1) AS merchant_ring_path, TO_JSON(p2) AS employer_ring_path
LIMIT 30;

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Step 4.2: Tabular Summary Ranking Delinquency Contagion & Collusion Rings Sharing Both Merchant AND Employer Segment
SELECT
  merchant_name,
  merchant_group,
  employer_segment_id,
  COUNT(*) AS connected_delinquent_pairs,
  COUNT(DISTINCT cif_1) + COUNT(DISTINCT cif_2) AS distinct_ring_members,
  ROUND(AVG(pair_unpaid_osp_myr), 2) AS avg_pair_unpaid_osp_myr,
  ROUND(SUM(pair_merchant_spend_myr), 2) AS total_ring_merchant_spend_myr
FROM GRAPH_TABLE(
  `acsm_gold.acsm_credit_ecosystem_graph`
  MATCH (c1:Customer)-[t1:TRANSACTED_AT]->(m:Merchant)<-[t2:TRANSACTED_AT]-(c2:Customer),
        (c1)-[w1:WORKS_IN_SEGMENT]->(e:EmployerSegment)<-[w2:WORKS_IN_SEGMENT]-(c2)
  WHERE c1.cif_id < c2.cif_id
    AND c1.is_delinquent = TRUE
    AND c2.is_delinquent = TRUE
  COLUMNS (
    c1.cif_id AS cif_1,
    c2.cif_id AS cif_2,
    m.merchant_name AS merchant_name,
    m.merchant_group AS merchant_group,
    e.employer_segment_id AS employer_segment_id,
    (c1.combined_unpaid_osp_myr + c2.combined_unpaid_osp_myr) AS pair_unpaid_osp_myr,
    (t1.total_spend_myr + t2.total_spend_myr) AS pair_merchant_spend_myr
  )
)
GROUP BY merchant_name, merchant_group, employer_segment_id
ORDER BY connected_delinquent_pairs DESC, avg_pair_unpaid_osp_myr DESC
LIMIT 15;

---
## Step 5 (Part A.4 — ISO GQL Query 3): AEON Ecosystem Cross-Sell Path Discovery (`Easy Payment` $\rightarrow$ `Credit Card`) (`%%bigquery --graph`)
ACSM's strategic **AEON 360** growth objective is converting reliable **Easy Payment (`EP`)** borrowers who actively shop at **AEON Privilege Merchants** into **AEON Credit Card (`CC`)** holders—while strictly respecting **Malaysian PDPA marketing consent (`pdpa_marketing_consent = 'Y'`)** and BNM responsible lending affordability (`new_dsr <= 50`, `is_delinquent = FALSE`).

Below, we traverse:
```
p = (f:CreditFacility {product_line: 'EP'}) <-[h:HOLDS_FACILITY]- (c:Customer {active_card_count: 0, is_delinquent: FALSE}) -[t:TRANSACTED_AT]-> (m:Merchant)
```
1. **Step 5.1 (`%%bigquery --graph`)** visualizes the **EP-to-CC Cross-Sell Paths (`TO_JSON(p) AS path`)** directly in the notebook graph widget so marketing and credit teams can visually inspect which `Merchant` hubs connect to multiple pre-qualified `EP` customers.
2. **Step 5.2 (`GRAPH_TABLE`)** outputs the prioritized tabular target list of pre-qualified **Customer Candidates** and **Merchant Locations** for instant Credit Card cross-sell at point-of-sale.

In [ ]:
%%bigquery --graph
-- Step 5.1: Interactive Graph Visualization of Pre-Qualified EP-to-CC Cross-Sell Paths
GRAPH acsm_gold.acsm_credit_ecosystem_graph
MATCH p = (f:CreditFacility)<-[h:HOLDS_FACILITY]-(c:Customer)-[t:TRANSACTED_AT]->(m:Merchant)
WHERE f.product_line = 'EP'
  AND c.active_card_count = 0
  AND c.is_delinquent = FALSE
  AND c.pdpa_marketing_consent = 'Y'
  AND c.net_income_myr >= 4000
  AND f.new_dsr <= 50
RETURN TO_JSON(p) AS path
LIMIT 40;

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Step 5.2: Tabular Ranking of Pre-Qualified EP-to-CC Cross-Sell Candidates via Graph Traversal
SELECT
  cif_id,
  customer_name,
  state,
  occupation,
  net_income_myr,
  ep_facility_id,
  ep_exposure_myr,
  ep_new_dsr,
  merchant_name,
  merchant_group,
  ep_merchant_spend_myr
FROM GRAPH_TABLE(
  `acsm_gold.acsm_credit_ecosystem_graph`
  MATCH (f:CreditFacility)<-[h:HOLDS_FACILITY]-(c:Customer)-[t:TRANSACTED_AT]->(m:Merchant)
  WHERE f.product_line = 'EP'
    AND c.active_card_count = 0
    AND c.is_delinquent = FALSE
    AND c.pdpa_marketing_consent = 'Y'
    AND c.net_income_myr >= 4000
    AND f.new_dsr <= 50
  COLUMNS (
    c.cif_id AS cif_id,
    c.customer_name AS customer_name,
    c.state AS state,
    c.occupation AS occupation,
    c.net_income_myr AS net_income_myr,
    f.facility_id AS ep_facility_id,
    f.exposure_amount_myr AS ep_exposure_myr,
    f.new_dsr AS ep_new_dsr,
    m.merchant_name AS merchant_name,
    m.merchant_group AS merchant_group,
    t.total_spend_myr AS ep_merchant_spend_myr
  )
)
ORDER BY ep_merchant_spend_myr DESC, net_income_myr DESC
LIMIT 20;

---
## Step 6: Cleanup Activity — Delete BigQuery Enterprise Reservation & Project Assignment
After completing the **BigQuery Property Graph (`GRAPH` & `GRAPH_TABLE` ISO GQL)** exercises, run the cleanup cell below to:
1. Delete the `QUERY` reservation assignment for `projects/{PROJECT_ID}`
2. Delete the `acsm-graph-enterprise-res` (`ENTERPRISE`, baseline `0`, max `500` slots) reservation so your project reverts to standard On-Demand query pricing.

In [ ]:
# @title Step 6.1: Cleanup — Unassign Project & Delete Enterprise Reservation (`acsm-graph-enterprise-res`)
%run aeon-credit-gcp-workshop/track1_platform_governance/scripts/bq_reservation_helper.py cleanup --project $PROJECT_ID --location $LOCATION --reservation-id acsm-graph-enterprise-res

---
## ✅ Track 1 (Notebook 05 — Lab 1.5) Summary

| Lab 1.5 Section | Capability Demonstrated | Key BigQuery Property Graph Artefacts (`asia-southeast1`) |
|---|---|---|
| **Step 1.0: Enterprise Slot Reservation** | Autoscaling Enterprise reservation (`baseline=0`, `max_slots=500`) assigned to project for `QUERY` execution | `bigqueryreservation.googleapis.com/v1` (`acsm-graph-enterprise-res` + `QUERY` assignment) |
| **Step 2: BigQuery Property Graph DDL** | Zero-ETL Property Graph over governed relational tables (`CREATE OR REPLACE PROPERTY GRAPH`) | `acsm_gold.acsm_credit_ecosystem_graph` (`Customer`, `CreditFacility`, `Merchant`, `EmployerSegment` nodes + `HOLDS_FACILITY`, `TRANSACTED_AT`, `WORKS_IN_SEGMENT` edges) |
| **Step 3: ISO GQL 2-Hop Portfolio Traversal (`%%bigquery --graph`)** | Interactive node-and-edge visualization (`RETURN TO_JSON(p) AS path`) + tabular `GRAPH_TABLE` aggregation across facilities, customers, and merchants | `p = (f:CreditFacility)<-[:HOLDS_FACILITY]-(c:Customer)-[:TRANSACTED_AT]->(m:Merchant)` |
| **Step 4: Fraud & Delinquency Contagion Rings (`%%bigquery --graph`)** | Interactive cyclical diamond visualization (`TO_JSON(p1)`, `TO_JSON(p2)`) + tabular ring ranking of delinquent customer pairs sharing both a merchant and employer segment | `(c1:Customer)-[:TRANSACTED_AT]->(m:Merchant)<-[:TRANSACTED_AT]-(c2:Customer)` + `WORKS_IN_SEGMENT` |
| **Step 5: AEON 360 Cross-Sell Path Discovery (`%%bigquery --graph`)** | Interactive path visualization (`RETURN TO_JSON(p) AS path`) + ranked target list of PDPA-consented, zero-delinquency Easy Payment customers transacting at privilege merchants without an active Credit Card | `GRAPH acsm_gold.acsm_credit_ecosystem_graph` filtered by `active_card_count = 0`, `is_delinquent = FALSE`, `pdpa_marketing_consent = 'Y'`, `new_dsr <= 50` |
| **Step 6: Reservation Cleanup** | Automated unassignment & deletion of the Enterprise reservation (`acsm-graph-enterprise-res`) | Reverts project to standard On-Demand query pricing |

➡️ **Next Lab**: Proceed to **[`06_BigQuery_Conversational_Analytics_Agent.ipynb`](./06_BigQuery_Conversational_Analytics_Agent.ipynb)** (**Lab 1.6**) to provision, publish, and chat with the governed **BigQuery Conversational Analytics (BQCA) Data Agent**!